# Biomarker Database Analyst Agent with Strands
This notebook demonstrates how to build a biomarker database analyst agent using open-source [Strands Agents](https://strandsagents.com/latest/) framework and how to deploy the agent on [Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/).

The agent will need the following tools:

- `get_schema`: Retrieves database schema information
- `query_redshift`: Executes SQL queries against the database
- `refine_sql`: Optimizes SQL queries for better performance

#### Install Strands agents and required dependencies

In [ ]:
%pip install --upgrade bedrock-agentcore-starter-toolkit --quiet

#### Ensure the latest version of boto3 is shown below
Ensure the boto3 version printed below is **1.39** or higher.

In [ ]:
%pip show boto3

#### Import required libraries

In [ ]:
import boto3
import json
import time
import uuid
from collections import defaultdict
from typing import Dict, Any
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore_starter_toolkit import Evaluation, Observability
from bedrock_agentcore.memory.client import MemoryClient

# Get AWS account information
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name

# Define Bedrock model id
MODEL_ID = "global.anthropic.claude-sonnet-4-6"

## Prerequisites

Run through the notebook environment setup in [00-setup_environment.ipynb](00-setup_environment.ipynb).

#### Setup AWS clients
Define the clients to AWS services that will be used by tools.

In [ ]:
# Initialize AWS clients
bedrock_client = boto3.client('bedrock-runtime', region_name=region)
redshift_client = boto3.client('redshift-data')

# Initialize AgentCore Memory for sharing query results between agents
memory_client = MemoryClient(region_name=region)
memory_resource = memory_client.create_or_get_memory(
    name="biomarker_agent_memory",
    strategies=[],
    description="Short-term memory for sharing query results between agents",
    event_expiry_days=3
)
memory_id = memory_resource.get("memoryId") or memory_resource.get("id")
memory_session_id = str(uuid.uuid4())
print(f"Memory ID: {memory_id}, Session ID: {memory_session_id}")

print(f"Region: {region}")
print(f"Account ID: {account_id}")

#### Cross-agent data sharing with AgentCore Memory

When running as part of a multi-agent system (like we'll do in notebook 05), the biomarker database analyst stores raw query results in [AgentCore Memory](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore-memory.html) after each Redshift query. This allows other sub-agents to retrieve the data directly from memory when performing additional analysis, eliminating the need for intermediate storage components like S3.

Memory events expire after 3 days, so there is no need to manage temporary files.

## Strands Agent Creation
In this section we create the agent using the Strands framework

#### Define agent configuration and instructions

In [ ]:
biomarker_agent_name = 'Biomarker-database-analyst-strands'
biomarker_agent_description = "biomarker query engine with redshift using Strands framework"
biomarker_agent_instruction = """
You are a medical research assistant AI specialized in generating SQL queries for a 
database containing medical biomarker information. Your primary task is to interpret user queries, 
generate appropriate SQL queries, and provide relevant medical insights based on the data. 
Use only the appropriate tools as required by the specific question. Follow these instructions carefully: 

1. Before generating any SQL query, use the get_schema tool to familiarize yourself with the database structure. 
This will ensure your queries are correctly formatted and target the appropriate columns. 

2. When generating an SQL query: 
   a. Write the query as a single line, removing all newline ("\n") characters. 
   b. Column names should remain consistent, do not modify the column names in the generated SQL query. 

3. Before execution of a step: 
   a. Evaluate the SQL query with the rationale of the specific step by using the refine_sql tool. 
      Provide both the SQL query and a brief rationale for the specific step you're taking. 
      Do not share the original user question with the tool. 
   b. Only proceed to execute the query using the query_redshift tool after receiving the evaluated 
      and potentially optimized version from the refine_sql tool. 
   c. If there is an explicit need for retrieving all the data in S3, avoid optimized query 
      recommendations that aggregate the data. 

4. When providing your response: 
   a. Start with a brief summary of your understanding of the user's query. 
   b. Explain the steps you're taking to address the query. 
   c. Ask for clarifications from the user if required.
"""

#### Define tools for Strands agent
These tools will invoke different services to perform operations for the agent

In [ ]:
def extract_table_columns(query):
    table_columns = defaultdict(list)
    for record in query["Records"]:
        table_name = record[0]["stringValue"]
        column_name = record[1]["stringValue"]
        column_type = record[2]["stringValue"]
        column_comment = record[3]["stringValue"]
        column_details = {
            "name": column_name,
            "type": column_type,
            "comment": column_comment
        }
        table_columns[table_name].append(column_details)
    return dict(table_columns)

# Define the tools using Strands @tool decorator
@tool
def get_schema() -> str:
    """
    Get the database schema including all table names and column information.
    This tool retrieves the structure of the redshift database to help formulate proper SQL queries.
    
    Returns:
        str: JSON string containing table names and their schemas
    """
    sql = """
        SELECT
            'clinical_genomic' AS table_name,
            a.attname AS column_name,
            pg_catalog.format_type(a.atttypid, a.atttypmod) AS column_type,
            pg_catalog.col_description(a.attrelid, a.attnum) AS column_comment
        FROM
            pg_catalog.pg_attribute a
        WHERE
            a.attrelid = 'clinical_genomic'::regclass
            AND a.attnum > 0
            AND NOT a.attisdropped;"""

    try:
        result = redshift_client.execute_statement(Database='dev', DbUser='admin', Sql=sql, ClusterIdentifier='biomarker-redshift-cluster')
    
        def wait_for_query_completion(statement_id):
            while True:
                response = redshift_client.describe_statement(Id=statement_id)
                status = response['Status']
                if status == 'FINISHED':
                    break
                elif status in ['FAILED', 'CANCELLED']:
                    print("SQL statement execution failed or was cancelled.")
                    break
                time.sleep(2)
        
        wait_for_query_completion(result['Id'])
        
        response = redshift_client.get_statement_result(Id=result['Id'])
        print(f"\nSchema Output: {str(response)[:500]}...\n")
        return response
    except Exception as e:
        print("Error:", e)
        raise

@tool
def query_redshift(query: str) -> str:
    """
    Execute a SQL query against the Redshift database.
    
    Args:
        query (str): The SQL query to execute
    
    Returns:
        str: Query results as JSON string
    """
    print(f"\nRedshift Input Query: {query}\n")
    try:
        result = redshift_client.execute_statement(Database='dev', DbUser='admin', Sql=query, ClusterIdentifier='biomarker-redshift-cluster')
    
        def wait_for_query_completion(statement_id):
            while True:
                response = redshift_client.describe_statement(Id=statement_id)
                status = response['Status']
                if status == 'FINISHED':
                    break
                elif status in ['FAILED', 'CANCELLED']:
                    print("SQL statement execution failed or was cancelled.")
                    break
                time.sleep(2)
        
        wait_for_query_completion(result['Id'])
        
        response = redshift_client.get_statement_result(Id=result['Id'])
        print(f"\nRedshift Output: {response}\n")

        # Store query results in AgentCore Memory for cross-agent access
        try:
            result_data = json.dumps({
                "ColumnMetadata": [{"name": col["name"]} for col in response.get("ColumnMetadata", [])],
                "Records": response.get("Records", [])
            }, default=str)
            memory_client.create_event(
                memory_id=memory_id,
                actor_id="biomarker_agent",
                session_id=memory_session_id,
                messages=[(result_data, "TOOL")],
                metadata={"type": {"stringValue": "query_results"}}
            )
            print(f"Query results stored in memory (session: {memory_session_id})")
        except Exception as mem_error:
            print(f"Warning: Failed to store results in memory: {mem_error}")

        return response
    except Exception as e:
        print("Error:", e)
        raise

@tool
def refine_sql(sql: str, question: str) -> str:
    """
    Evaluate and potentially optimize an SQL query for efficiency.
    
    Args:
        sql (str): The SQL query to evaluate
        question (str): The rationale or step description for this query
    
    Returns:
        str: Evaluated/optimized SQL query
    """
    print(f"\nInput SQL: {sql}, Input Question: {question}\n")
    raw_schema = get_schema()
    schema = extract_table_columns(raw_schema)

    prompt = f"""
    You are an extremely critical SQL query evaluation assistant. Your job is to analyze
    the given schema, SQL query, and question to ensure the query is efficient and accurately answers the 
    question. You should focus on making the query as efficient as possible, using aggregation when applicable.

    Here is the schema you should consider:
    <schema>
    {json.dumps(schema)}
    </schema>
    
    Pay close attention to the accepted values and the column data type located in the comment field for each column.
    
    Here is the generated SQL query to evaluate:
    <sql_query>
    {sql}
    </sql_query>
    
    Here is the question that was asked:
    <question>
    {question}
    </question>
    
    Your task is to evaluate and refine the SQL query to ensure it is very efficient. Follow these steps:
    1. Analyze the query in relation to the schema and the question.
    2. Determine if the query efficiently answers the question.
    3. If the query is not efficient, provide a more efficient SQL query.
    4. If the query is already efficient, respond with "no change needed".

    When evaluating efficiency, consider the following:
    - Use of appropriate aggregation functions (COUNT, SUM, AVG, etc.)
    - Proper use of GROUP BY clauses
    - Avoiding unnecessary JOINs or subqueries
    - Selecting only necessary columns
    - Using appropriate WHERE clauses to filter data
    
    Here are examples to guide your evaluation:
    
    Inefficient query example:
    SELECT chemotherapy, survival_status FROM dev.public.lung_cancer_cases WHERE chemotherapy = 'Yes';

    This is inefficient because it does not provide a concise and informative output that directly answers
    the question. It results in a larger output size, does not aggregate the data, and presents the results
    in a format that is not easy to analyze and interpret.

    Efficient query example:
    SELECT survival_status, COUNT(*) AS count FROM dev.public.lung_cancer_cases WHERE chemotherapy = 'Yes' GROUP BY survival_status;

    This query uses COUNT(*) and GROUP BY to aggregate and count the records for each distinct value of survival_status, providing a more concise and informative result.
    
    Another efficient query example:
    SELECT smoking_status, COUNT(DISTINCT case_id) AS num_patients FROM clinical_genomic WHERE age_at_histological_diagnosis > 50 GROUP BY smoking_status;
    
    This query uses COUNT(DISTINCT) and GROUP BY to aggregate and provide a summary of the data, reducing the SQL output size.
    
    If you suggest a new query, do not use line breaks in the generated SQL. Your response should be a single line of SQL or "no change needed" if the original query is already efficient.
    
    Remember to prioritize aggregation when possible to reduce SQL output size and provide more meaningful results.
    """
    
    try:
        user_message = {"role": "user", "content": prompt}
        claude_response = {"role": "assistant", "content": ""}
        model_Id = MODEL_ID
        messages = [user_message, claude_response]
        system_prompt = "You are an extremely critical sql query evaluation assistant, your job is to look at the schema, sql query and question being asked to then evaluate the query to ensure it is efficient."
        max_tokens = 1000
        
        body = json.dumps({
            "messages": messages,
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "system": system_prompt
        })
    
        response = bedrock_client.invoke_model(body=body, modelId=model_Id)
        response_bytes = response.get("body").read()
        response_text = response_bytes.decode('utf-8')
        response_json = json.loads(response_text)
        content = response_json.get('content', [])
        for item in content:
            if item.get('type') == 'text':
                result_text = item.get('text')
                print(f"\nRefined SQL: {result_text}\n")
                return result_text
        return "No SQL found in response"
    except Exception as e:
        print("Error:", e)
        raise

# Create list of tools
biomarker_agent_tools = [get_schema, query_redshift, refine_sql]
print(f"Created {len(biomarker_agent_tools)} tools for the Strands agent")

#### Setup AWS Bedrock provider for Strands

In [ ]:
# Create Bedrock model for Strands
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    temperature=0.1,
    streaming=True
)

#### Create the Strands agent

In [ ]:
# Create the Strands agent
try:
    biomarker_agent = Agent(
        model=model,
        tools=biomarker_agent_tools,
        system_prompt=biomarker_agent_instruction
    )
    
    print("Successfully created Strands agent")
    print(f"Agent has {len(biomarker_agent_tools)} tools available")
    
except Exception as e:
    print(f"Error creating agent: {e}")
    raise

#### Test the agent locally

In [ ]:
# Test the agent with a simple query
test_query = "How many patients are current smokers?"

print(f"Testing agent with query: {test_query}")
print("=" * 64)

try:
    # Run the agent
    response = biomarker_agent(test_query)
    
except Exception as e:
    print(f"Error during agent execution: {e}")
    import traceback
    traceback.print_exc()

#### Advanced usage examples

In [ ]:
# Example of more complex queries
complex_queries = [
    "What is the average age of patients with lung cancer?",
    "Show me the distribution of biomarker levels by cancer stage"
]

def test_complex_query(query: str):
    """
    Test a complex query with the agent
    """
    print(f"\nTesting query: {query}")
    print("-" * 75)
    
    try:
        response = biomarker_agent(query)
    except Exception as e:
        print(f"Error: {e}")

for query in complex_queries: 
    test_complex_query(query)

#### Session management and conversation continuity

In [ ]:
# Demonstrate conversation continuity
def interactive_session():
    """
    Simple interactive session with the agent
    """
    print("Interactive Biomarker Database Analysis Session")
    print("Type 'quit' to exit")
    print("=" * 50)
    
    while True:
        user_input = input("\nYour question: ")
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Session ended.")
            break
            
        try:
            response = biomarker_agent(user_input)
            if response.tool_calls:
                print(f"\n[Used {len(response.tool_calls)} tool(s)]")    
        except Exception as e:
            print(f"Error: {e}")

interactive_session()

## Deploy agent on AgentCore
In this section we are going to deploy the biomarker analyst agent to [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) Runtime.

### Preparing your agent for deployment on AgentCore Runtime

In [ ]:
%%writefile biomarker_agent_runtime.py

import json
import uuid
import strands
from strands import Agent
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from biomarker_agent import *

# Create the Strands agent
try:
    agent = Agent(
        model=model,
        tools=biomarker_agent_tools,
        system_prompt=biomarker_agent_instruction
    )
    
    print("Successfully created Strands agent")
    print(f"Agent has {len(biomarker_agent_tools)} tools available")
    
except Exception as e:
    print(f"Error creating agent: {e}")
    raise

# Custom event serializer function
def json_serializer(obj):
    if isinstance(obj, strands.agent.agent.Agent):
        return obj.name
    elif isinstance(obj, strands.agent.agent_result.AgentResult):
        result = {}
        result["message"] = str(obj)
        result["metrics"] = {}
        result["metrics"]["accumulated_usage"] = obj.metrics.accumulated_usage
        result["metrics"]["accumulated_metrics"] = obj.metrics.accumulated_metrics
        return result
    elif isinstance(obj, uuid.UUID):
        return str(obj)
    else:
        # Assume other objects are not serializable
        return str(obj)

app = BedrockAgentCoreApp()

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    Invoke the agent with streaming capabilities
    This function demonstrates how to implement streaming responses
    with AgentCore Runtime using async generators
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    
    try:
        # Stream each chunk as it becomes available
        async for event in agent.stream_async(user_input):
            print("Received event:", event)
            # serialize the event object
            yield json.dumps(event, default=json_serializer)
    except Exception as e:
        # Handle errors gracefully in streaming context
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response

if __name__ == "__main__":
    app.run()

### Deploying the agent to AgentCore Runtime

#### Define agent name and retrieve runtime role

In [ ]:
from utils.boto3_helper import get_role_arn
iam = boto3.client('iam')

agent_name="biomarker_agent"
agentcore_iam_role = get_role_arn('BedrockAgentCoreStrands')
agentcore_iam_role

#### Configure AgentCore Runtime deployment
During the configure step, your docker file will be generated based on your application code.

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

# Remove existing Dockerfile so configure() regenerates it with the correct entrypoint
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="biomarker_agent_runtime.py",
    execution_role=agentcore_iam_role,
    auto_create_ecr=True,
    requirements_file="runtime_requirements.txt",
    region=region,
    agent_name=agent_name
)
response

#### Launching agent to AgentCore Runtime
Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

In [ ]:
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True
)
launch_result

#### Invoking AgentCore Runtime
Finally, we can invoke our AgentCore Runtime with a payload to test our agent. What you see here is the raw response in JSON format. We'll see how to parse the response in the next example.

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "How many patients are current smokers?"})
invoke_response

### Invoking AgentCore Runtime with boto3
Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import json
import boto3
import uuid
from IPython.display import Markdown, display
from botocore.config import Config

session_id = str(uuid.uuid4())
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region,
    config=Config(read_timeout=600, connect_timeout=10)
)

test_query = "How many patients are current smokers?"
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": test_query}),
    runtimeSessionId=session_id
)

print(f"Response contentType: {response.get("contentType")}\n")
print(f"Testing agent: {test_query}")
print("=" * (15 + len(test_query)))

if "text/event-stream" in response.get("contentType", ""):
    # Processing streaming response
    for line in response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                # remove the SSE structure
                data = line[6:]
                # we need to parse it twice to convert from JSON str to a dictionary
                data_obj = json.loads(data)
                data_obj = json.loads(data_obj)
                # for this example we only care about the data field
                if "data" in data_obj:
                    print(data_obj.get("data"), end="", flush=True)
    print()  # final newline after streaming completes
else:
    # Handle non-streaming response
    try:
        events = []
        for event in response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    if events:
        try:
            response_data = json.loads(events[0].decode("utf-8"))
            display(Markdown(response_data))
        except:
            print(f"Raw response: {events[0]}")

## [Optional] Evaluate Agent

We are going to use AgentCore on-demand built-in evaluators to analyze our agent. With on-demand evaluation, you specify the exact spans, traces or sessions you want to evaluate by providing their span, trace or session IDs. When using the AgentCore Starter toolkit you can also automatically evaluate all traces in a session.

#### Initialize AgentCore Evaluations Client

Now let's initiate the AgentCore Evaluations client from the AgentCore Starter toolkit.

In [ ]:
eval_client = Evaluation(region=region)

# extract agent id from agent arn
agent_id = agent_arn.rsplit('/', 1)[-1]

You can use the list_evaluators() function to see a list of build in evaluators.

In [ ]:
available_evaluators = eval_client.list_evaluators()

#### Wait for Trace Ingestion

The observability service (CloudWatch) takes some time to ingest traces and spans after agent invocations. If we run evaluations immediately, some traces may not be available yet, leading to incomplete or missing evaluation results.

The cell below uses the Observability client to poll for traces until all expected traces from our test questions are available.



In [ ]:
import time

obs_client = Observability(agent_id=agent_id, region=region)
expected_traces = 1
max_wait = 180  # maximum seconds to wait
poll_interval = 15  # seconds between checks
elapsed = 0

print(f"Waiting for {expected_traces} traces to be available in the observability service...\n")

while elapsed < max_wait:
    try:
        trace_data = obs_client.list(session_id=session_id)
        available = len(trace_data.traces) if trace_data and trace_data.traces else 0
        print(f"Found {available}/{expected_traces} traces ({elapsed}s elapsed)")

        if available >= expected_traces:
            print("\nAll traces are available. Proceeding with evaluations.")
            break
    except Exception as e:
        print(f"Waiting for traces... ({elapsed}s elapsed) - {e}")

    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    print(f"\nTimed out after {max_wait}s. Proceeding with {available if 'available' in dir() else 'unknown'} traces."
          " Some evaluation results may be incomplete.")

#### Running Evaluations

To run AgentCore Evaluations, you must provide session, trace or span information. Different metrics require different level of information from your agent traces, as we saw in the table with built-in evaluations.

##### Trace level metrics
- **Builtin.Coherence:** Evaluates whether the response is logically structured and coherent.
- **Builtin.Conciseness:** Evaluates whether the response is appropriately brief without missing key information.
- **Builtin.Correctness:** Evaluates whether the information in the agent's response is factually accurate.
- **Builtin.InstructionFollowing:** Measures how well the agent follows the provided system instructions.

In [ ]:
trace_level_results = eval_client.run(
    agent_id=agent_id,
    session_id=session_id, 
    evaluators=["Builtin.Coherence", "Builtin.Conciseness", "Builtin.Correctness", "Builtin.InstructionFollowing"]
)

##### Session level metrics

- **Builtin.GoalSuccessRate:** Evaluates whether the conversation successfully meets the user's goals.



In [ ]:
goal_sucess_results = eval_client.run(
    agent_id=agent_id,
    session_id=session_id, 
    evaluators=["Builtin.GoalSuccessRate"]
)

##### Tool level metrics

- **Builtin.ToolSelectionAccuracy:** Component Level Metric. Evaluates whether the agent selected the appropriate tool for the task.

In [ ]:
tool_selection_results = eval_client.run(
    agent_id=agent_id,
    session_id=session_id, 
    evaluators=["Builtin.ToolSelectionAccuracy"]
)

### Analyzing Evaluation Results

Let's now analyze the results. In this case, we are evaluating the session with two different metrics in the same run. That means that we now need to know which evaluator is producing each response. We can do that with the evaluator_name property of the result. Let's see how well our agent used tools:

In [ ]:
for result in trace_level_results.results:
    if result.label != None:
        information = f"""
        {result.evaluator_name} Result: {result.label} ({result.value})
        Explanation: \n{result.explanation}]\n
        Token Usage: {result.token_usage}\n
        Context: {result.context}\n
        """
        print("===================================================")
        display(Markdown(information))

In [ ]:
for result in goal_sucess_results.results:
    information = f"""
    {result.evaluator_name} Result: {result.label} ({result.value})
    Explanation: \n{result.explanation}]\n
    Token Usage: {result.token_usage}\n
    Context: {result.context}\n
    """
    print("===================================================")
    display(Markdown(information))

In [ ]:
for result in tool_selection_results.results:
    if result.label != None:
        information = f"""
        {result.evaluator_name} Result: {result.label} ({result.value})
        Explanation: \n{result.explanation}]\n
        Token Usage: {result.token_usage}\n
        Context: {result.context}\n
        """
        print("===================================================")
        display(Markdown(information))